# Resident Satisfaction Analysis

Aggregates survey feedback scores by site and date, weights response types, and tracks
satisfaction trends over time to surface properties with significant declines.

Stack: PySpark, Microsoft Fabric lakehouse.

Note: internal identifiers are anonymized, and outputs are cleared because the originals
ran on internal data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pyspark.sql.functions import col, when
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from datetime import datetime

# Load Data

## Ticekts

In [ ]:
# Load the processed data
tickets = spark.sql("SELECT * FROM Silver_Lakehouse.rp.maintenance_tickets_processed")

# Load sentiment data
sentiment = spark.sql("SELECT * FROM Gold_AnalyticalData.Gold_Lakehouse.rp.survey_sentiment_score")

# Join data
tickets = tickets.join(sentiment, tickets.id == sentiment.event_id, how='left').drop('event_id')

In [ ]:
# Data preparation
df = tickets.filter(col('survey_date').isNotNull())
df = df.sort(col('survey_date'))
df = df.withColumn('survey_date', col('survey_date').cast('date'))
df = df.withColumn('happy', when(col('happy') == True, 1).otherwise(0))
df = df.toPandas()

# Aggregating the feedback scores

In [ ]:
# Define weights
happy_weight = 0.4
recommend_weight = 0.2
sentiment_weight = 0.4

# Adjust weights if sentiment score is missing
df['Adjusted_Happy_Weight'] = df['Sentiment_Score'].isnull().map({True: 0.6, False: happy_weight})
df['Adjusted_Recommend_Weight'] = df['Sentiment_Score'].isnull().map({True: 0.4, False: recommend_weight})
df['Adjusted_Sentiment_Weight'] = df['Sentiment_Score'].notnull().astype(float) * sentiment_weight

# Normalize weights so they sum to 1
df['Total_Weight'] = (
    df['Adjusted_Happy_Weight'] + 
    df['Adjusted_Recommend_Weight'] + 
    df['Adjusted_Sentiment_Weight']
)
df['Normalized_Happy_Weight'] = df['Adjusted_Happy_Weight'] / df['Total_Weight']
df['Normalized_Recommend_Weight'] = df['Adjusted_Recommend_Weight'] / df['Total_Weight']
df['Normalized_Sentiment_Weight'] = df['Adjusted_Sentiment_Weight'] / df['Total_Weight']

# Normalize individual metrics
df['Recommend_Score_Norm'] = (df['recommendation'] - 1) / 9  # Normalize to [0, 1]
df['Sentiment_Score_Norm'] = (df['Sentiment_Score'] + 1) / 2  # Normalize to [0, 1]

# Calculate Aggregated Score with adjusted weights
df['Aggregated_Score'] = (
    df['happy'] * df['Normalized_Happy_Weight'] +
    df['Recommend_Score_Norm'] * df['Normalized_Recommend_Weight'] +
    df['Sentiment_Score_Norm'].fillna(0) * df['Normalized_Sentiment_Weight']  # Use 0 for missing sentiment
)

# Trend Analysis

In [ ]:
# Aggregate by site and date, counting happy responses and total responses
df_grouped = df.groupby(['site', 'survey_date']).agg(
    Happy_Sum=('happy', 'sum'),
    Total_Responses_Happy=('happy', 'count'),
    Recommend_Sum=('recommendation', 'sum'),
    Total_Responses_Recommend=('recommendation', 'count'),
    Sentiment_Sum=('Sentiment_Score', lambda x: np.nan if x.isna().all() else x.sum()),
    Total_Responses_Sentiment=('Sentiment_Score', 'count'),
    Aggregated_Sum=('Aggregated_Score', 'sum'),
    Total_Aggregated_Responses=('Aggregated_Score', 'count'),
).reset_index()
df_grouped = df_grouped.sort_values(by=['site', 'survey_date'])
df_grouped['survey_date'] = pd.to_datetime(df_grouped['survey_date'])

# Set the date as the index for rolling calculations
df_grouped.set_index('survey_date', inplace=True)

# Determine the overall earliest and latest dates across all sites
global_start_date = df_grouped.index.min()
global_end_date = df_grouped.index.max()
common_date_range = pd.date_range(start=global_start_date, end=global_end_date, freq='D')

# Initialize an empty list to store reindexed data
df_aggregated_full = []

for site in df_grouped['site'].unique():
    # Filter the data for the specific site
    df_site = df_grouped[df_grouped['site'] == site].copy()
    
    # Determine the start and end date specific to this site
    site_start_date = df_site.index.min()
    site_end_date = df_site.index.max()
    
    # Create a date range only within the valid period for this site
    site_date_range = pd.date_range(start=site_start_date, end=site_end_date, freq='D')
    
    # Reindex the data to ensure it matches the site-specific date range
    df_site = df_site.reindex(site_date_range)
    
    # Fill the 'site' column for newly created rows
    df_site['site'] = site
    
    # Append the processed site data
    df_aggregated_full.append(df_site)

# Combine all the reindexed data into a single DataFrame
df_aggregated = pd.concat(df_aggregated_full)

# Reset the index to ensure proper handling of date columns
df_aggregated.reset_index(inplace=True)
df_aggregated.rename(columns={'index': 'survey_date'}, inplace=True)
df_aggregated.set_index('survey_date', inplace=True)

# Rolling Windows
short_win = '60D'
long_win = '730D'


####################
## 1) Happy Ratio
# Calculate rolling sum of happy responses and total responses over the short window
df_aggregated['Rolling_Happy_Sum'] = df_aggregated.groupby('site')['Happy_Sum'].rolling(short_win, min_periods=5).sum().reset_index(level=0, drop=True)
df_aggregated['Rolling_Total_Responses_Happy'] = df_aggregated.groupby('site')['Total_Responses_Happy'].rolling(short_win, min_periods=5).sum().reset_index(level=0, drop=True)

# Calculate the proportion of happy responses in the short-term window
df_aggregated['Happy_Ratio_SMA'] = df_aggregated['Rolling_Happy_Sum'] / df_aggregated['Rolling_Total_Responses_Happy']

# Calculate the long-term average and standard deviation
df_aggregated['Happy_Ratio_LMA'] = df_aggregated.groupby('site')['Happy_Ratio_SMA'].rolling(long_win, min_periods=30).mean().reset_index(level=0, drop=True)
df_aggregated['Happy_Ratio_std'] = df_aggregated.groupby('site')['Happy_Ratio_SMA'].rolling(long_win, min_periods=30).std().reset_index(level=0, drop=True)

# Calculate the Z-score for detecting anomalies
df_aggregated['Z_Score_happy'] = (df_aggregated['Happy_Ratio_SMA'] - df_aggregated['Happy_Ratio_LMA']) / df_aggregated['Happy_Ratio_std']

# Identify anomalies based on Z-score
anomaly_threshold = -2  # Adjust this threshold as needed
df_aggregated['Happy_Anomaly'] = (df_aggregated['Z_Score_happy'] < anomaly_threshold).astype(int)


#####################
## 2) Recommend Score
# Calculate rolling sum of Recommend scores and total responses over the short window
df_aggregated['Rolling_Recommend_Sum'] = df_aggregated.groupby('site')['Recommend_Sum'].rolling(short_win, min_periods=5).sum().reset_index(level=0, drop=True)
df_aggregated['Rolling_Total_Responses_Recommend'] = df_aggregated.groupby('site')['Total_Responses_Recommend'].rolling(short_win, min_periods=5).sum().reset_index(level=0, drop=True)

# Calculate the proportion of happy responses in the short-term window
df_aggregated['Recommend_Ratio_SMA'] = df_aggregated['Rolling_Recommend_Sum'] / df_aggregated['Rolling_Total_Responses_Recommend']

# Calculate the long-term average and standard deviation
df_aggregated['Recommend_Ratio_LMA'] = df_aggregated.groupby('site')['Recommend_Ratio_SMA'].rolling(long_win, min_periods=30).mean().reset_index(level=0, drop=True)
df_aggregated['Recommend_Ratio_std'] = df_aggregated.groupby('site')['Recommend_Ratio_SMA'].rolling(long_win, min_periods=30).std().reset_index(level=0, drop=True)

# Calculate the Z-score for detecting anomalies
df_aggregated['Z_Score_recommend'] = (df_aggregated['Recommend_Ratio_SMA'] - df_aggregated['Recommend_Ratio_LMA']) / df_aggregated['Recommend_Ratio_std']

# Identify anomalies based on Z-score
anomaly_threshold = -2  # Adjust this threshold as needed
df_aggregated['Recommend_Anomaly'] = (df_aggregated['Z_Score_recommend'] < anomaly_threshold).astype(int)


#####################
## 3) Sentiment Score
# Calculate rolling sum of Recommend scores and total responses over the short window
df_aggregated['Rolling_Sentiment_Sum'] = df_aggregated.groupby('site')['Sentiment_Sum'].rolling(short_win, min_periods=5).sum().reset_index(level=0, drop=True)
df_aggregated['Rolling_Total_Responses_Sentiment'] = df_aggregated.groupby('site')['Total_Responses_Sentiment'].rolling(short_win, min_periods=5).sum().reset_index(level=0, drop=True)

# Calculate the proportion of happy responses in the short-term window
df_aggregated['Sentiment_Ratio_SMA'] = df_aggregated['Rolling_Sentiment_Sum'] / df_aggregated['Rolling_Total_Responses_Sentiment']

# Calculate the long-term average and standard deviation
df_aggregated['Sentiment_Ratio_LMA'] = df_aggregated.groupby('site')['Sentiment_Ratio_SMA'].rolling(long_win, min_periods=30).mean().reset_index(level=0, drop=True)
df_aggregated['Sentiment_Ratio_std'] = df_aggregated.groupby('site')['Sentiment_Ratio_SMA'].rolling(long_win, min_periods=30).std().reset_index(level=0, drop=True)

# Calculate the Z-score for detecting anomalies
df_aggregated['Z_Score_sentiment'] = (df_aggregated['Sentiment_Ratio_SMA'] - df_aggregated['Sentiment_Ratio_LMA']) / df_aggregated['Sentiment_Ratio_std']

# Identify anomalies based on Z-score
anomaly_threshold = -2  # Adjust this threshold as needed
df_aggregated['Sentiment_Anomaly'] = (df_aggregated['Z_Score_sentiment'] < anomaly_threshold).astype(int)

#####################
## 3) Aggregated Score
# Calculate rolling sum of Aggregated Score and total responses over the short window
df_aggregated['Rolling_Aggregated_Sum'] = df_aggregated.groupby('site')['Aggregated_Sum'].rolling(short_win, min_periods=5).sum().reset_index(level=0, drop=True)
df_aggregated['Rolling_Total_Responses_Aggregated'] = df_aggregated.groupby('site')['Total_Aggregated_Responses'].rolling(short_win, min_periods=5).sum().reset_index(level=0, drop=True)

# Calculate the proportion of happy responses in the short-term window
df_aggregated['Aggregated_Score_SMA'] = df_aggregated['Rolling_Aggregated_Sum'] / df_aggregated['Rolling_Total_Responses_Aggregated']

# Calculate the long-term average and standard deviation
df_aggregated['Aggregated_Score_LMA'] = df_aggregated.groupby('site')['Aggregated_Score_SMA'].rolling(long_win, min_periods=30).mean().reset_index(level=0, drop=True)
df_aggregated['Aggregated_Score_std'] = df_aggregated.groupby('site')['Aggregated_Score_SMA'].rolling(long_win, min_periods=30).std().reset_index(level=0, drop=True)

# Calculate the Z-score for detecting anomalies
df_aggregated['Z_Score_Aggregated'] = (df_aggregated['Aggregated_Score_SMA'] - df_aggregated['Aggregated_Score_LMA']) / df_aggregated['Aggregated_Score_std']

# Identify anomalies based on Z-score
anomaly_threshold = -2  # Adjust this threshold as needed
df_aggregated['Aggregated_Anomaly'] = (df_aggregated['Z_Score_Aggregated'] < anomaly_threshold).astype(int)


# Aggregating the results
#df_aggregated.set_index('date', inplace=True)
# Sort the sites with the highest number of consecutive anomalies
flagged_sites = []
for site in df_aggregated['site'].unique():
    df_site = df_aggregated.query('site == @site')
    df_site = df_site.loc[df_site.index >= (df_site.index.max() - pd.DateOffset(months=12))]
    flagged_sites.append({'site': site,
                          'sum_z_sentiment': df_site['Z_Score_sentiment'].sum(),
                          'sum_z_recommend': df_site['Z_Score_recommend'].sum(),
                          'sum_z_happy': df_site['Z_Score_happy'].sum(),
                          'sum_z_aggregated': df_site['Z_Score_Aggregated'].sum()})
                          
flagged_sites = pd.DataFrame(flagged_sites)

flagged_sites['satisfaction_index'] = (flagged_sites['sum_z_aggregated'])


# Calculating the Z-Score of the Area Under the Curve
mean_area = flagged_sites['satisfaction_index'].mean()
std_area = flagged_sites['satisfaction_index'].std()
flagged_sites['satisfaction_index_z_score'] = (flagged_sites['satisfaction_index'] - mean_area) / std_area


flagged_sites.sort_values('satisfaction_index_z_score',inplace=True)
flagged_sites.reset_index(inplace=True, drop=True)


## Updating the lakehouse

In [ ]:
from datetime import datetime

# Get the current date
current_date = datetime.now()

flagged_sites['date'] = current_date

# melting the table
satisfaction_summary = flagged_sites.melt(id_vars=['site', 'date'],
                                                  value_vars=['satisfaction_index', 'satisfaction_index_z_score'],
                                                  var_name='attribute', 
                                                  value_name='value'
                                                  )

satisfaction_summary_spark = spark.createDataFrame(satisfaction_summary)

# Save the DataFrame as a table in the specified schema (append mode)
satisfaction_summary_spark.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save("abfss://Gold_AnalyticalData@onelake.dfs.fabric.microsoft.com/Gold_Lakehouse.Lakehouse/Tables/rp/satisfaction_trends_summary")                                                

In [ ]:
cols = ['survey_date', 'site', 
        'Happy_Ratio_SMA', 'Happy_Ratio_LMA', 'Happy_Anomaly',
        'Recommend_Ratio_SMA', 'Recommend_Ratio_LMA', 'Recommend_Anomaly',
        'Sentiment_Ratio_SMA', 'Sentiment_Ratio_LMA', 'Sentiment_Anomaly',
        'Aggregated_Score_SMA', 'Aggregated_Score_LMA', 'Aggregated_Anomaly']
satisfaction_trends = df_aggregated.reset_index('survey_date')

satisfaction_trends_spark = spark.createDataFrame(satisfaction_trends[cols])


# Save the DataFrame as a table in the specified schema (append mode)
satisfaction_trends_spark.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save("abfss://Gold_AnalyticalData@onelake.dfs.fabric.microsoft.com/Gold_Lakehouse.Lakehouse/Tables/rp/satisfaction_trends")

In [ ]:
# import matplotlib.pyplot as plt
# import matplotlib.dates as mdates
# from datetime import datetime, timedelta

# # Filter data for the specific site
# site = '55113'
# df_site = df_aggregated[(df_aggregated['site'] == site)]
# df_site = df_site.reset_index('survey_date')
# df_site.rename(columns={'survey_date':'date'}, inplace=True)
# df_site = df_site[df_site['date'].dt.year >= 2022]

# # Calculate the date 12 months ago from the most recent date in the dataset
# last_date = df_site['date'].max()
# start_of_last_12_months = last_date - timedelta(days=365)

# # Create subplots with shared x-axis
# fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(25, 12), sharex=True)

# # Plot Happy Ratio
# ax1.plot(df_site['date'], df_site['Happy_Ratio_SMA'], marker='o', linestyle='-', color='blue', label='Happy Ratio SMA', alpha=0.2)
# ax1.plot(df_site['date'], df_site['Happy_Ratio_LMA'], marker='_', linestyle='-', color='green', label='Happy Ratio LMA')

# # Fill the area between SMA and LMA where SMA is below LMA
# ax1.fill_between(df_site['date'], df_site['Happy_Ratio_SMA'], df_site['Happy_Ratio_LMA'], 
#                  where=(df_site['Happy_Ratio_SMA'] < df_site['Happy_Ratio_LMA']), 
#                  color='red', alpha=0.3, label='Area Below LMA')

# ax1.fill_between(df_site['date'], df_site['Happy_Ratio_SMA'], df_site['Happy_Ratio_LMA'], 
#                  where=(df_site['Happy_Ratio_SMA'] > df_site['Happy_Ratio_LMA']), 
#                  color='green', alpha=0.3, label='Area Above LMA')

# ax1.scatter(df_site['date'][df_site['Happy_Anomaly'] == 1], 
#             df_site['Happy_Ratio_SMA'][df_site['Happy_Anomaly'] == 1],
#             color='red', s=100, edgecolor='red', label='Anomalies')

# # Add a vertical line for the start of the last 12 months
# ax1.axvline(start_of_last_12_months, color='black', linestyle='--', linewidth=2, label='Start of Last 12 Months')
# ax1.set_ylim(0, 1)

# ax1.set_title(f'Trend of Happiness Ratio Over Time for Site {site}')
# ax1.set_ylabel('Happiness Ratio')
# ax1.legend(loc='lower left')
# ax1.grid(True)


# # Plot Recommend Score
# ax2.plot(df_site['date'], df_site['Recommend_Ratio_SMA'], marker='o', linestyle='-', color='blue', label='Recommend Score SMA', alpha=0.2)
# ax2.plot(df_site['date'], df_site['Recommend_Ratio_LMA'], marker='_', linestyle='-', color='green', label='Recommend Score LMA')

# # Fill the area between SMA and LMA where SMA is below LMA
# ax2.fill_between(df_site['date'], df_site['Recommend_Ratio_SMA'], df_site['Recommend_Ratio_LMA'], 
#                  where=(df_site['Recommend_Ratio_SMA'] < df_site['Recommend_Ratio_LMA']), 
#                  color='red', alpha=0.3, label='Area Below LMA')

# ax2.fill_between(df_site['date'], df_site['Recommend_Ratio_SMA'], df_site['Recommend_Ratio_LMA'], 
#                  where=(df_site['Recommend_Ratio_SMA'] > df_site['Recommend_Ratio_LMA']), 
#                  color='green', alpha=0.3, label='Area Above LMA')

# ax2.scatter(df_site['date'][df_site['Recommend_Anomaly'] == 1], 
#             df_site['Recommend_Ratio_SMA'][df_site['Recommend_Anomaly'] == 1],
#             color='red', s=100, edgecolor='red', label='Anomalies')

# # Add a vertical line for the start of the last 12 months
# ax2.axvline(start_of_last_12_months, color='black', linestyle='--', linewidth=2, label='Start of Last 12 Months')

# ax2.set_ylim(1, 10)
# ax2.set_title(f'Trend of Average Recommend Score Over Time for Site {site}')
# ax2.set_ylabel('Average Recommend Score')
# ax2.legend(loc='lower left')
# ax2.grid(True)

# # Plot Sentiment Score
# ax3.plot(df_site['date'], df_site['Sentiment_Ratio_SMA'], marker='o', linestyle='-', color='blue', label='Sentiment Score SMA', alpha=0.2)
# ax3.plot(df_site['date'], df_site['Sentiment_Ratio_LMA'], marker='_', linestyle='-', color='green', label='Sentiment Score LMA')

# # Fill the area between SMA and LMA where SMA is below LMA
# ax3.fill_between(df_site['date'], df_site['Sentiment_Ratio_SMA'], df_site['Sentiment_Ratio_LMA'], 
#                  where=(df_site['Sentiment_Ratio_SMA'] < df_site['Sentiment_Ratio_LMA']), 
#                  color='red', alpha=0.3, label='Area Below LMA')

# ax3.fill_between(df_site['date'], df_site['Sentiment_Ratio_SMA'], df_site['Sentiment_Ratio_LMA'], 
#                  where=(df_site['Sentiment_Ratio_SMA'] > df_site['Sentiment_Ratio_LMA']), 
#                  color='green', alpha=0.3, label='Area Above LMA')

# ax3.scatter(df_site['date'][df_site['Sentiment_Anomaly'] == 1], 
#             df_site['Sentiment_Ratio_SMA'][df_site['Sentiment_Anomaly'] == 1],
#             color='red', s=100, edgecolor='red', label='Anomalies')

# # Add a vertical line for the start of the last 12 months
# ax3.axvline(start_of_last_12_months, color='black', linestyle='--', linewidth=2, label='Start of Last 12 Months')

# ax3.set_ylim(-1, 1)
# ax3.set_title(f'Trend of Average Sentiment Score Over Time for Site {site}')
# ax3.set_xlabel('Date')
# ax3.set_ylabel('Average Sentiment Score')
# ax3.legend(loc='lower left')
# ax3.grid(True)

# # Plot Aggrgated Score
# ax4.plot(df_site['date'], df_site['Aggregated_Score_SMA'], marker='o', linestyle='-', color='blue', label='Aggregated Score SMA', alpha=0.2)
# ax4.plot(df_site['date'], df_site['Aggregated_Score_LMA'], marker='_', linestyle='-', color='green', label='Aggregated Score LMA')

# # Fill the area between SMA and LMA where SMA is below LMA
# ax4.fill_between(df_site['date'], df_site['Aggregated_Score_SMA'], df_site['Aggregated_Score_LMA'], 
#                  where=(df_site['Aggregated_Score_SMA'] < df_site['Aggregated_Score_LMA']), 
#                  color='red', alpha=0.3, label='Area Below LMA')

# ax4.fill_between(df_site['date'], df_site['Aggregated_Score_SMA'], df_site['Aggregated_Score_LMA'], 
#                  where=(df_site['Aggregated_Score_SMA'] > df_site['Aggregated_Score_LMA']), 
#                  color='green', alpha=0.3, label='Area Above LMA')

# ax4.scatter(df_site['date'][df_site['Aggregated_Anomaly'] == 1], 
#             df_site['Aggregated_Score_SMA'][df_site['Aggregated_Anomaly'] == 1],
#             color='red', s=100, edgecolor='red', label='Anomalies')

# # Add a vertical line for the start of the last 12 months
# ax4.axvline(start_of_last_12_months, color='black', linestyle='--', linewidth=2, label='Start of Last 12 Months')

# ax4.set_ylim(0, 1)
# ax4.set_title(f'Trend of Feedback Aggregated Score Over Time for Site {site}')
# ax4.set_ylabel('Aggregate Score')
# ax4.legend(loc='lower left')
# ax4.grid(True)

# # Set the x-axis to display quarterly intervals for both plots
# ax4.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
# ax4.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# # Display the plot
# plt.tight_layout()
# plt.show()

# Last Year Data

In [ ]:
df['survey_date'] = pd.to_datetime(df['survey_date'])
df.sort_values(by=['survey_date'], inplace=True)
df['survey_date'] = df['survey_date'].dt.date

# Filtering the last year data
last_day = df['survey_date'].max()
one_year_ago = (pd.Timestamp(last_day) - pd.DateOffset(years=1)).date()
df_last_year = df[df['survey_date'] >= one_year_ago]

site_sentiment = df_last_year.groupby('site').aggregate(response_count = ('id', 'count'),
                                              happiness_ratio=('happy', 'mean'),
                                              avg_recommend_score = ('recommendation', 'mean'),
                                              avg_sentiment_score = ('Sentiment_Score', 'mean'),
                                              avg_aggregate_score = ('Aggregated_Score', 'mean')).reset_index()  

# the sites with less than 10 response are assigned NaN for the following metric
site_sentiment.loc[site_sentiment['response_count'] < 10, 'happiness_ratio'] = np.nan
site_sentiment.loc[site_sentiment['response_count'] < 10, 'avg_recommend_score'] = np.nan
site_sentiment.loc[site_sentiment['response_count'] < 10, 'avg_sentiment_score'] = np.nan
site_sentiment.loc[site_sentiment['response_count'] < 10, 'avg_aggregate_score'] = np.nan

# calculating z-score
site_sentiment['happy_ratio_z_score'] = (site_sentiment['happiness_ratio'] - site_sentiment['happiness_ratio'].mean())/site_sentiment['happiness_ratio'].std()
site_sentiment['recommend_z_score'] = (site_sentiment['avg_recommend_score'] - site_sentiment['avg_recommend_score'].mean())/site_sentiment['avg_recommend_score'].std()
site_sentiment['sentiment_z_score'] = (site_sentiment['avg_sentiment_score'] - site_sentiment['avg_sentiment_score'].mean())/site_sentiment['avg_sentiment_score'].std()
site_sentiment['combined_score'] = (site_sentiment['avg_aggregate_score'] - site_sentiment['avg_aggregate_score'].mean())/site_sentiment['avg_aggregate_score'].std()
 
site_sentiment.query('combined_score<-2').sort_values('combined_score')

In [ ]:
from datetime import datetime

# Get the current date
current_date = datetime.now()

site_sentiment['date'] = current_date

# melting the table
last_12_months_satisfaction = site_sentiment.melt(id_vars=['site', 'date'],
                                                  value_vars=['response_count', 'happiness_ratio', 'avg_recommend_score', 
                                                  'avg_sentiment_score', 'happy_ratio_z_score', 'recommend_z_score', 
                                                  'sentiment_z_score', 'avg_aggregate_score','combined_score'],
                                                  var_name='attribute', 
                                                  value_name='value'
                                                  )

last_12_months_satisfaction_spark = spark.createDataFrame(last_12_months_satisfaction)

# Ensure the 'date' column is of the correct type
last_12_months_satisfaction_spark = last_12_months_satisfaction_spark.withColumn("date", last_12_months_satisfaction_spark["date"].cast("date"))

# Save the DataFrame as a table in the specified schema (append mode)
last_12_months_satisfaction_spark.write.mode("append").format("delta").save("abfss://Gold_AnalyticalData@onelake.dfs.fabric.microsoft.com/Gold_Lakehouse.Lakehouse/Tables/rp/last_12_months_satisfaction")                                                

In [ ]:
# # Create a dot plot
# score = 'combined_score'
# # Sort the sites by Under-Curve Area
# site_sentiment = site_sentiment.sort_values(score)

# # Generate random x positions with jitter for each site
# x_positions = np.random.uniform(-0.8, 0.8, size=len(site_sentiment))

# # Create the plot
# plt.figure(figsize=(12, 1))
# plt.scatter(site_sentiment[score], x_positions, color='blue', alpha=0.2)

# # Highlight flagged sites
# for i, (index, row) in enumerate(site_sentiment.iterrows()):
#     if row[score] < -2:
#         plt.scatter(row[score], x_positions[i], color='red', s=50)
#         plt.text(row[score]+.22, x_positions[i]-.1, f"{row['site']}", color='red', ha='center', fontsize=8)

# # Add labels and title
# plt.ylim(-1, 1)
# plt.xlim(-4, 4)
# # remove x tick labels
# plt.yticks([])
# plt.xlabel('Satisfaction Index')

# # Display the plot
# plt.show()

# site_sentiment.query('combined_score<-2').sort_values('combined_score')[['name', 'Region', 'City', 'asset_class','Suite Count','response_count','happiness_ratio', 'avg_recommend_score', 'avg_sentiment_score', 'combined_score']].round(2).reset_index(drop=True)